# Review-to-Revenue: Data Foundation

This notebook creates the Unity Catalog tables that power our Restaurant Analytics Agent.

**Data pipeline**: Ingest compressed Google Local data → filter San Diego Italian restaurants → create Delta tables in Unity Catalog


## Table of Contents

### Data Engineering
1. **Setup Unity Catalog Structure** - Create catalog, schema, and volume for data storage
   - 1.1. Inspect Staged Files - Verify data quality and schema

2. **Filter San Diego Italian Restaurants** - Apply geographic and category filters

3. **Join Reviews to Filtered Restaurants** - Create reviews dataset

4. **Create Restaurant Metrics Table** - Aggregate review data for Business Lookup and Competitor Comparison tools

5. **Prepare for Vector Search (Review Text)** - Stage review text data for Theme Retrieval tool

6. **Build Vector Search Assets** - Create and initialize the vector-search-ready source table

## Pipeline outputs

- `main.aai510_final_agent.restaurants`
- `main.aai510_final_agent.reviews`
- `main.aai510_final_agent.restaurant_metrics`
- `main.aai510_final_agent.reviews_with_text`
- `main.aai510_final_agent.reviews_for_vector_search`
- Vector Search index: `main.aai510_final_agent.restaurant_review_text_index`


In [0]:
# Install required packages
%pip install databricks-vectorsearch
%pip install databricks-vectorsearch openai
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.10.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5ea4965e-0fb5-4c0c-9ba5-b6ced7d58904
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Not uninstalling requests at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5ea4965e-0fb5-4c0c-9ba5-b6ced7d58904
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.3
    Not uninstalling protobuf at /databricks/python3/lib/python3.1

---
## 1. Setup Unity Catalog Structure

Create the catalog, schema, and volume for staging source data and storing agent tables.

**UC namespace**: `main.aai510_final_agent`  
**Volume**: Stores compressed source files for reusable Spark reads (no local `wget` downloads)  
**Source data**:
- `meta-California.json.gz` - Business metadata (name, address, categories, location)
- `rating-California.csv.gz` - User ratings (gmap_id, user_id, rating, timestamp)

In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "aai510_final_agent"
VOLUME = "aai510_final_raw"

# Source URLs
META_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/meta-California.json.gz"
RATING_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/rating-California.csv.gz"

# Create Unity Catalog objects
print("=" * 60)
print("Creating Unity Catalog structure...")
print("=" * 60)

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Volume paths for staged files
META_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/meta-California.json.gz"
RATING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/rating-California.csv.gz"

# Helper function to check whether files already exist in Volume
def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

print(f"\nStaging compressed files to Volume: {CATALOG}.{SCHEMA}.{VOLUME}")
print("Checking for existing staged files...\n")

# Stage files into Volume only if needed
if file_exists(META_PATH):
    print(f"✓ Already staged: {META_PATH}")
else:
    dbutils.fs.cp(META_URL, META_PATH)
    print(f"✓ Staged: {META_PATH}")

if file_exists(RATING_PATH):
    print(f"✓ Already staged: {RATING_PATH}")
else:
    dbutils.fs.cp(RATING_URL, RATING_PATH)
    print(f"✓ Staged: {RATING_PATH}")

print(f"\n✓ Unity Catalog setup complete")
print(f"\tCatalog: {CATALOG}")
print(f"\tSchema: {CATALOG}.{SCHEMA}")
print(f"\tVolume: {CATALOG}.{SCHEMA}.{VOLUME}")

Creating Unity Catalog structure...

Staging compressed files to Volume: main.aai510_final_agent.aai510_final_raw
Checking for existing staged files...

✓ Already staged: /Volumes/main/aai510_final_agent/aai510_final_raw/meta-California.json.gz
✓ Already staged: /Volumes/main/aai510_final_agent/aai510_final_raw/rating-California.csv.gz

✓ Unity Catalog setup complete
	Catalog: main
	Schema: main.aai510_final_agent
	Volume: main.aai510_final_agent.aai510_final_raw


---
### 1.1. Inspect Staged Files

Verify that files were successfully staged and examine their schemas to understand the data structure.

**What we're checking**:
- Total record counts for California dataset
- Schema structure (column names and types)
- Sample records to confirm data quality
- Whether review text is available (needed for Vector Search)

This step helps identify any data quality issues early and confirms which agent tools can be implemented with the current dataset.

In [0]:
# Verify staged files and inspect schema
print("=" * 60)
print("Inspecting Staged Files...")
print("=" * 60)

# Read metadata sample
print("\n1. Business Metadata (meta-California.json.gz)")
print("-" * 60)
meta_df = spark.read.json(META_PATH)
print(f"Total businesses in California: {meta_df.count():,}")
print("\nSample records:")
display(meta_df.limit(3))

# Read ratings sample
print("\n2. Ratings Data (rating-California.csv.gz)")
print("-" * 60)
ratings_df = spark.read.option("header", "true").csv(RATING_PATH)
print(f"Total ratings in California: {ratings_df.count():,}")
print("\nSample records:")
display(ratings_df.limit(3))

Inspecting Staged Files...

1. Business Metadata (meta-California.json.gz)
------------------------------------------------------------
Total businesses in California: 515,961

Sample records:


MISC,address,avg_rating,category,description,gmap_id,hours,latitude,longitude,name,num_of_reviews,price,relative_results,state,url
null,"City Textile, 3001 E Pico Blvd, Los Angeles, CA 90023",4.5,List(Textile exporter),null,0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,null,34.0188913,-118.2152898,City Textile,6,null,"List(0x80c2c624136ea88b:0xb0315367ed448771, 0x80c2c97cb7c52f17:0xb66ee68c1c366f2d)",Open now,https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c98c0e3c16fd:0x29ec8a728764fdf9?authuser=-1&hl=en&gl=us
"List(List(Wheelchair accessible entrance), null, List(Good for kids), List(Casual), null, null, null, null, null, null, null, null, List(Comfort food), null, null, null, null, List(Takeout, Dine-in, Delivery))","San Soo Dang, 761 S Vermont Ave, Los Angeles, CA 90005",4.4,List(Korean restaurant),null,0x80c2c778e3b73d33:0xbdc58662a4a97d49,"List(List(Thursday, 6:30AM–6PM), List(Friday, 6:30AM–6PM), List(Saturday, 6:30AM–6PM), List(Sunday, 7AM–12PM), List(Monday, Closed), List(Tuesday, 6:30AM–6PM), List(Wednesday, 6:30AM–6PM))",34.0580917,-118.2921295,San Soo Dang,18,null,"List(0x80c2c78249aba68f:0x35bf16ce61be751d, 0x80c2c79998f75fff:0xd7ca5c67e96fb778, 0x80c2b899146d7507:0xf4162c12c9cf65f8, 0x80c2c77f2d419951:0x26285631b21e324c, 0x80c2b8add9016015:0x15836f81a963b35f)",Open ⋅ Closes 6PM,https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c778e3b73d33:0xbdc58662a4a97d49?authuser=-1&hl=en&gl=us
"List(null, null, null, null, null, null, null, null, null, null, null, null, null, List(Checks, Debit cards, Credit cards), null, null, null, List(In-store shopping))","Nova Fabrics, 2200 E 11th St, Los Angeles, CA 90021",3.3,List(Fabric store),null,0x80c2c89923b27a41:0x32041559418d447,"List(List(Thursday, 9AM–5PM), List(Friday, 9AM–5PM), List(Saturday, Closed), List(Sunday, Closed), List(Monday, 9AM–5PM), List(Tuesday, 9AM–5PM), List(Wednesday, 9AM–5PM))",34.0236689,-118.2329297,Nova Fabrics,6,null,"List(0x80c2c8811477253f:0x23a8a492df1918f7, 0x80c2c6333ea5aaa7:0x38c93218b0d5751b, 0x80c2c633235effa1:0x384bf96fe70cbbc7, 0x80c2c6295991d031:0x8516999f61103f87, 0x80c2c63331db7ec5:0x507f59d850d36691)",Open ⋅ Closes 5PM,https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c89923b27a41:0x32041559418d447?authuser=-1&hl=en&gl=us



2. Ratings Data (rating-California.csv.gz)
------------------------------------------------------------
Total ratings in California: 70,082,062

Sample records:


business,user,rating,timestamp
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,113165551130476225599,5,1599164133778
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,101226371370637614545,5,1618261672851
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,111167703666981482712,5,1524515066787


---
## 2. Filter San Diego Italian Restaurants

Read business metadata and filter for San Diego Italian restaurants using Spark SQL expressions. This creates the foundational `restaurants` table that powers the Business Lookup tool and defines which reviews are relevant for analysis.

**Filters applied**:
- **Geography**: Address contains "San Diego, CA"
- **Category**: Business category includes "Restaurant" AND "Italian"

**Output table**: `main.aai510_final_agent.restaurants`

In [0]:
# Read and filter business metadata
print("=" * 60)
print("Filtering San Diego Italian restaurants...")
print("=" * 60)

# Read metadata with schema inference
meta_df = spark.read.json(META_PATH)

# Filter logic (mimics original regex + category matching)
restaurants_df = (
    meta_df
    # Convert category array to searchable string
    .withColumn("category_text", 
                F.lower(F.concat_ws(" ", F.coalesce(F.col("category"), F.array()))))
    # Normalize address for matching
    .withColumn("address_lower", 
                F.lower(F.coalesce(F.col("address"), F.lit(""))))
    # Apply filters
    .filter(F.col("address_lower").rlike(r",\s*san diego,\s*ca\b"))
    .filter(F.col("category_text").contains("restaurant"))
    .filter(F.col("category_text").contains("italian"))
    # Select clean schema for agent tools
    .select(
        "gmap_id",
        F.col("name").alias("business_name"),
        "address",
        "category",
        "latitude",
        "longitude"
    )
    .dropna(subset=["gmap_id"])
    .dropDuplicates(["gmap_id"])
)

# Preview results
restaurant_count = restaurants_df.count()
print(f"\n✓ Found {restaurant_count:,} San Diego Italian restaurants")

print("\nSample restaurants:")
display(restaurants_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurants"

print("=" * 60)
print(f"\nWriting to Unity Catalog: {table_name}")
print("=" * 60)

restaurants_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

Filtering San Diego Italian restaurants...

✓ Found 199 San Diego Italian restaurants

Sample restaurants:


gmap_id,business_name,address,category,latitude,longitude
0x80d95337fe3c340d:0x9f53399ee78194a5,Monzu Fresh Pasta,"Monzu Fresh Pasta, 455 Tenth Ave, San Diego, CA 92101",List(Italian restaurant),32.709932699999996,-117.1553409
0x80dc00de8e439425:0xc5a831ef8ec36b4e,Round Table Pizza,"Round Table Pizza, 3250 Governor Dr, San Diego, CA 92122","List(Pizza restaurant, Caterer, Italian restaurant, Pizza delivery, Pizza Takeout)",32.8524068,-117.215419
0x80d954e6cf09735d:0x198b17b146574c05,Pizzeria Bruno Napoletano,"Pizzeria Bruno Napoletano, 4207 Park Blvd, San Diego, CA 92103, United States","List(Pizza restaurant, Italian restaurant)",32.7542463,-117.14610719999999
0x80dbf0d2060249c3:0x59adb765ada8a513,Round Table Pizza,"Round Table Pizza, 16761 Bernardo Center Dr, San Diego, CA 92128","List(Pizza restaurant, Caterer, Italian restaurant, Pizza delivery, Pizza Takeout)",33.0171972,-117.07439749999999
0x80d955f154df0ff1:0x54aaa39afb13d877,De Niro Ristorante,"De Niro Ristorante, 4768 Convoy St, San Diego, CA 92111","List(Italian restaurant, Restaurant)",32.8283801,-117.15379259999999



Writing to Unity Catalog: main.aai510_final_agent.restaurants
✓ Table created: main.aai510_final_agent.restaurants


---
## 3. Join Reviews to Filtered Restaurants

Read the ratings CSV and join to our filtered San Diego Italian restaurants. This creates the `reviews` table that contains all customer ratings for our target restaurants.

**Output table**: `main.aai510_final_agent.reviews`  
**Schema**: `gmap_id`, `user_id`, `rating`, `timestamp`

**Note**: This dataset contains ratings only (no review text). For Vector Search and Complaint Extraction tools, review text will need to be sourced separately.

In [0]:
# Read and join reviews to filtered restaurants
print("=" * 60)
print("Loading and filtering reviews...")
print("=" * 60)

# Read ratings CSV with header
ratings_df = (
    spark.read
    .option("header", "true")
    .csv(RATING_PATH)
    .withColumnRenamed("business", "gmap_id")
    .withColumnRenamed("user", "user_id")
)

print(f"Total ratings in California: {ratings_df.count():,}")

# Load restaurant filter list
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
restaurant_ids = restaurants_df.select("gmap_id")

print(f"Filtering to {restaurant_ids.count():,} San Diego Italian restaurants...")

# Inner join to get only reviews for our target restaurants
reviews_df = (
    ratings_df
    .join(restaurant_ids, on="gmap_id", how="inner")
    .select("gmap_id", "user_id", "rating", "timestamp")
)

review_count = reviews_df.count()
print(f"\n✓ Found {review_count:,} reviews for San Diego Italian restaurants")

print("\nSample reviews:")
display(reviews_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.reviews"
print(f"\nWriting to Unity Catalog: {table_name}")
reviews_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

# Summary stats
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Restaurants: {restaurant_ids.count():,}")
print(f"Reviews: {review_count:,}")
print(f"Avg reviews per restaurant: {review_count / restaurant_ids.count():.1f}")

Loading and filtering reviews...
Total ratings in California: 70,082,062
Filtering to 199 San Diego Italian restaurants...

✓ Found 82,845 reviews for San Diego Italian restaurants

Sample reviews:


gmap_id,user_id,rating,timestamp
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,114976159291119258205,1,1564203402789
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,100305772822037390573,4,1501468574646
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,110966308572404885088,5,1552792044271
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,115555920222063468568,4,1556292850077
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,101357089882070703342,5,1557093995582



Writing to Unity Catalog: main.aai510_final_agent.reviews
✓ Table created: main.aai510_final_agent.reviews

SUMMARY
Restaurants: 199
Reviews: 82,845
Avg reviews per restaurant: 416.3


---
## 4. Create Restaurant Metrics Table

Aggregate review data to create metrics for each restaurant. This table powers the **Business Lookup** and **Competitor Comparison** agent tools.

**Output table**: `main.aai510_final_agent.restaurant_metrics`  
**Metrics computed**:
- Average rating
- Total review count
- Rating distribution (count by 1-5 stars)
- Latest review timestamp
- Restaurant metadata (name, address, location)

In [0]:
# Create aggregated metrics from reviews
print("=" * 60)
print("Computing restaurant metrics...")
print("=" * 60)

# Load reviews and restaurants
reviews_df = spark.table(f"{CATALOG}.{SCHEMA}.reviews")
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")

# Aggregate metrics per restaurant
metrics_df = (
    reviews_df
    .groupBy("gmap_id")
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.count("*").alias("review_count"),
        F.max("timestamp").alias("latest_review_timestamp"),
        # Rating distribution: count by rating value (1-5 stars)
        F.sum(F.when(F.col("rating") == "1", 1).otherwise(0)).alias("rating_1_count"),
        F.sum(F.when(F.col("rating") == "2", 1).otherwise(0)).alias("rating_2_count"),
        F.sum(F.when(F.col("rating") == "3", 1).otherwise(0)).alias("rating_3_count"),
        F.sum(F.when(F.col("rating") == "4", 1).otherwise(0)).alias("rating_4_count"),
        F.sum(F.when(F.col("rating") == "5", 1).otherwise(0)).alias("rating_5_count"),
    )
)

# Join with restaurant metadata
restaurant_metrics_df = (
    metrics_df
    .join(restaurants_df, on="gmap_id", how="inner")
    .select(
        "gmap_id",
        "business_name",
        "address",
        "category",
        "latitude",
        "longitude",
        "avg_rating",
        "review_count",
        "latest_review_timestamp",
        "rating_1_count",
        "rating_2_count",
        "rating_3_count",
        "rating_4_count",
        "rating_5_count",
    )
    .orderBy(F.desc("review_count"))
)

print(f"\n✓ Computed metrics for {restaurant_metrics_df.count():,} restaurants")

print("\nTop 10 restaurants by review count:")
display(restaurant_metrics_df.limit(10))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
print(f"\nWriting to Unity Catalog: {table_name}")
restaurant_metrics_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

print("\n" + "=" * 60)
print("✓ Restaurant metrics table ready for agent tools")
print("=" * 60)

Computing restaurant metrics...

✓ Computed metrics for 199 restaurants

Top 10 restaurants by review count:


gmap_id,business_name,address,category,latitude,longitude,avg_rating,review_count,latest_review_timestamp,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count
0x80d954b2060ef165:0xc0de3f7db480b86,Filippi's Pizza Grotto Little Italy,"Filippi's Pizza Grotto Little Italy, 1747 India St, San Diego, CA 92101","List(Italian restaurant, Caterer, Pizza restaurant, Sandwich shop, Wine bar)",32.7237596,-117.1681514,4.483330194010171,5309,1618418280662,162,131,352,998,3666
0x80d954d0c6b6ef1f:0xd1f0c147aed157b4,Bronx Pizza,"Bronx Pizza, 111 Washington St, San Diego, CA 92103","List(Pizza restaurant, Italian restaurant, Restaurant)",32.7497524,-117.1634461,4.735729386892178,3311,1619755100238,40,34,93,427,2717
0x80d9535a475211fb:0x16d2e94fee4a042f,The Old Spaghetti Factory,"The Old Spaghetti Factory, 275 Fifth Ave, San Diego, CA 92101","List(Italian restaurant, Banquet hall, Bar, Delivery service, Family restaurant, Gluten-free restaurant, Takeout Restaurant, Restaurant)",32.708209,-117.15995199999999,4.318061088977424,3012,1618800096527,99,110,291,746,1766
0x80d954b3b5f8140f:0xd692822c3ed8379c,Mona Lisa Italian Foods,"Mona Lisa Italian Foods, 2061 India St, San Diego, CA 92101","List(Italian restaurant, Deli, Family restaurant, Pizza restaurant, Restaurant, Wine store)",32.7262337,-117.16907459999999,4.58653149891383,2762,1621915018849,58,52,131,492,2029
0x80d9537e305d3911:0x2108857782e4a0ce,Buona Forchetta - South Park,"Buona Forchetta - South Park, 3001 Beech St, San Diego, CA 92102","List(Pizza restaurant, Italian restaurant, Pasta shop, Restaurant)",32.721163499999996,-117.13003859999999,4.668550873586844,1946,1619496062516,33,35,72,264,1542
0x80deaae4ea5276a3:0x5406b47f1937c678,Olive Garden Italian Restaurant,"Olive Garden Italian Restaurant, 3215 Sports Arena Blvd, San Diego, CA 92110","List(Italian restaurant, Caterer, Family restaurant, Gluten-free restaurant, Takeout Restaurant, Seafood restaurant, Soup restaurant, Wine bar)",32.752378,-117.20728799999999,4.299789251844047,1898,1619048951164,87,64,163,463,1121
0x80d954b21a99cba9:0x76e8c703d80d71f4,Civico 1845,"Civico 1845, 1845 India St, San Diego, CA 92101",List(Italian restaurant),32.724289999999996,-117.168351,4.458424507658643,1828,1619639046264,62,55,112,353,1246
0x80d954ade7b8cbe7:0x65851f19969f4d14,Buon Appetito Restaurant,"Buon Appetito Restaurant, 1609 India St, San Diego, CA 92101","List(Italian restaurant, Restaurant, Southern Italian restaurant)",32.7224015,-117.1681218,4.518742442563482,1654,1618515764686,45,45,90,301,1173
0x80dbf9fed5bf4b2f:0x4cf3d5b15706bb2c,Olive Garden Italian Restaurant,"Olive Garden Italian Restaurant, 11555 Carmel Mountain Rd, San Diego, CA 92128","List(Italian restaurant, Caterer, Family restaurant, Gluten-free restaurant, Takeout Restaurant, Seafood restaurant, Soup restaurant, Wine bar)",32.978521,-117.082449,4.235013623978202,1468,1618785137043,51,50,152,465,750
0x80d954ae6f3567db:0x40ff164aecf06ffb,Pappalecco,"Pappalecco, 1602 State St, San Diego, CA 92101","List(Italian restaurant, Cafe, Coffee shop, Ice cream shop)",32.722077999999996,-117.1666587,4.664051355206848,1402,1618676897575,12,20,69,225,1076



Writing to Unity Catalog: main.aai510_final_agent.restaurant_metrics
✓ Table created: main.aai510_final_agent.restaurant_metrics

✓ Restaurant metrics table ready for agent tools


---
## 5. Prepare for Vector Search (Review Text)

The current `reviews` table only contains ratings (1-5 stars) without review text. For **Vector Search** and **Complaint/Opportunity Extraction** tools, we need the actual review text to create embeddings.

**Requirements for Vector Search**:
- Review text field for GTE Large embeddings
- Join reviews to filtered San Diego Italian restaurants
- Create Databricks Vector Search index

**Data source**: `review-California.json.gz` contains full review data including text field

In [0]:
# Prepare Review Text for Theme Retrieval

from pyspark.sql import functions as F

# Review text configuration
REVIEW_TEXT_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/review-California.json.gz"
REVIEW_TEXT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/review-California.json.gz"
REVIEW_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"

print("=" * 60)
print("Preparing review text for Theme Retrieval...")
print("=" * 60)

print(f"\nReview text source: {REVIEW_TEXT_URL}")
print(f"Review text path: {REVIEW_TEXT_PATH}")
print(f"Review text table: {REVIEW_TEXT_TABLE}")

Preparing review text for Theme Retrieval...

Review text source: https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/review-California.json.gz
Review text path: /Volumes/main/aai510_final_agent/aai510_final_raw/review-California.json.gz
Review text table: main.aai510_final_agent.reviews_with_text


In [0]:
# Check whether review text table already exists
try:
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Found existing table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))
    
    REVIEW_TEXT_READY = True

except Exception:
    print(f"Review text table not found: {REVIEW_TEXT_TABLE}")
    print("Checking whether raw review text file is already staged...")
    
    REVIEW_TEXT_READY = False

✓ Found existing table: main.aai510_final_agent.reviews_with_text
✓ Total review-text rows: 50,113


gmap_id,user_id,rating,time,text,name
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,114976159291119258205,1,1564203402789,No atmosphere. No fresh ingredients. No variety. The salads ordered has browning lettuce leaves. The fettuccine chicken was completely void of flavor and soggy from the diluted amounts of water. The sandwich ordered was slapped together. Where’s Gordon Ramsey?,Evelynn Herrera
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,100305772822037390573,4,1501468574646,Traveled from mesa Arizona. was delicious even though we came in a little before closing they still let us sit and eat. Loved the environment and scene. Will definitely come back again.,chris bilagody
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,110966308572404885088,5,1552792044271,"It is and always has been a wonderful place. Family owned and operated, I have been coming here with my family for 50 years!! They may finally close in September you must check it out if you can. It is a legend to experience at least once. Johnny still plays music at 91. Amazing💝",denise moore
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,115555920222063468568,4,1556292850077,"Family run for families and music fans. 66 years and closing this September. No more strolling accordion playing, groups singing along, neighborhood restaurant. Not for hip, fast food, quiet, and Yelp critique. It will get low marks. But a last stand for honor, families, hard work, dedication, and community. Proud to have memories.",Suz She
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,101357089882070703342,5,1557093995582,"There aren't many restaurants that have been around so long and have maintained their original charm as Pernicano's. You will enjoy the family atmosphere, decor that spans decades, and authentic dishes. Parking can be a challenge at busier times, but it's worth the extra effort.",Thom Hiatt


In [0]:
# Stage raw review text file only if needed
if not REVIEW_TEXT_READY:
    
    try:
        dbutils.fs.ls(REVIEW_TEXT_PATH)
        print(f"✓ Already staged: {REVIEW_TEXT_PATH}")
        
    except Exception:
        print("Raw review text file not found.")
        print("Staging raw review text file. This may take several minutes...")
        
        dbutils.fs.cp(REVIEW_TEXT_URL, REVIEW_TEXT_PATH)
        
        print(f"✓ Staged: {REVIEW_TEXT_PATH}")

else:
    print("Skipping raw file staging because review text table already exists.")

Skipping raw file staging because review text table already exists.


In [0]:
# Create review text table only if needed
if not REVIEW_TEXT_READY:
    
    print("=" * 60)
    print("Creating reviews_with_text table...")
    print("=" * 60)
    
    # Read raw review text file
    reviews_with_text_raw_df = spark.read.json(REVIEW_TEXT_PATH)
    
    # Get filtered restaurant IDs from Section 2
    restaurant_ids_df = (
        spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
        .select("gmap_id")
        .distinct()
    )
    
    # Keep only written reviews for selected restaurants
    reviews_with_text_filtered_df = (
        reviews_with_text_raw_df
        .join(restaurant_ids_df, on="gmap_id", how="inner")
        .select(
            "gmap_id",
            "user_id",
            "rating",
            "time",
            "text",
            "name"
        )
        .filter(F.col("text").isNotNull())
        .filter(F.length(F.col("text")) > 10)
    )
    
    # Save filtered review text table
    (
        reviews_with_text_filtered_df
        .write
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(REVIEW_TEXT_TABLE)
    )
    
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Created table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))

else:
    print("Skipping table creation because review text table already exists.")

Skipping table creation because review text table already exists.


---
## 6. Build Vector Search Assets

This section creates the vector-search-ready source table, initializes the Vector Search client, and creates or syncs the index. 

In [0]:
from pyspark.sql import functions as F

# Source tables
RESTAURANT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
RESTAURANTS_TABLE = f"{CATALOG}.{SCHEMA}.restaurants"
REVIEWS_WITH_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"
REVIEWS_FOR_VECTOR_TABLE = f"{CATALOG}.{SCHEMA}.reviews_for_vector_search"

print("=" * 60)
print("Configuring agent tool source tables...")
print("=" * 60)

print(f"Restaurant metrics table: {RESTAURANT_METRICS_TABLE}")
print(f"Restaurants table: {RESTAURANTS_TABLE}")
print(f"Review text table: {REVIEWS_WITH_TEXT_TABLE}")
print(f"Vector search source table: {REVIEWS_FOR_VECTOR_TABLE}")

Configuring agent tool source tables...
Restaurant metrics table: main.aai510_final_agent.restaurant_metrics
Restaurants table: main.aai510_final_agent.restaurants
Review text table: main.aai510_final_agent.reviews_with_text
Vector search source table: main.aai510_final_agent.reviews_for_vector_search


### 6.1. Prepare Review Text Table for Vector Search

Vector Search needs a stable primary key. This creates a new table with a `review_id`

To keep vector search efficient while making the agent more generalizable, this section indexes all available review text for the two selected comparison restaurants and a random sample of 20 additional San Diego Italian restaurants. This allows the agent to answer the primary comparison question while still supporting theme retrieval for other restaurants in the dataset.

In [0]:
# Create a vector-search-ready review text table with a stable primary key
# Include the two selected restaurants plus a random sample of other restaurants

from pyspark.sql.window import Window

SELECTED_RESTAURANTS = ["cesarina", "parma"]
NUM_RANDOM_RESTAURANTS = 20
MAX_REVIEWS_PER_RANDOM_RESTAURANT = 50

reviews_with_text_df = spark.table(REVIEWS_WITH_TEXT_TABLE)
restaurants_df = spark.table(RESTAURANTS_TABLE)

reviews_joined_df = (
    reviews_with_text_df
    .join(
        restaurants_df.select("gmap_id", "business_name", "address"),
        on="gmap_id",
        how="inner"
    )
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.col("text")) > 10)
)

# Keep all reviews for selected restaurants
selected_reviews_df = (
    reviews_joined_df
    .filter(
        (F.lower(F.col("business_name")).contains("cesarina")) |
        (F.lower(F.col("business_name")).contains("parma"))
    )
)

# Randomly choose additional restaurants, excluding selected restaurants
random_restaurant_ids_df = (
    reviews_joined_df
    .filter(
        ~(
            (F.lower(F.col("business_name")).contains("cesarina")) |
            (F.lower(F.col("business_name")).contains("parma"))
        )
    )
    .select("gmap_id", "business_name")
    .distinct()
    .orderBy(F.rand(seed=42))
    .limit(NUM_RANDOM_RESTAURANTS)
)

# For those random restaurants, take up to N reviews per restaurant
random_reviews_base_df = (
    reviews_joined_df
    .join(
        random_restaurant_ids_df.select("gmap_id"),
        on="gmap_id",
        how="inner"
    )
)

window_by_restaurant = Window.partitionBy("gmap_id").orderBy(F.rand(seed=42))

random_reviews_df = (
    random_reviews_base_df
    .withColumn("random_review_rank", F.row_number().over(window_by_restaurant))
    .filter(F.col("random_review_rank") <= MAX_REVIEWS_PER_RANDOM_RESTAURANT)
    .drop("random_review_rank")
)

# Combine selected restaurants and random sample
reviews_for_vector_df = (
    selected_reviews_df
    .unionByName(random_reviews_df)
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.col("gmap_id"),
                F.coalesce(F.col("user_id"), F.lit("unknown_user")),
                F.coalesce(F.col("time").cast("string"), F.lit("unknown_time")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        ).alias("review_id"),
        "gmap_id",
        "business_name",
        "address",
        "rating",
        "time",
        "text"
    )
    .dropDuplicates(["review_id"])
)

(
    reviews_for_vector_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(REVIEWS_FOR_VECTOR_TABLE)
)

# Change Data Feed is required for Delta Sync Vector Search indexes
spark.sql(f"""
ALTER TABLE {REVIEWS_FOR_VECTOR_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"Created vector-search source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Total rows: {spark.table(REVIEWS_FOR_VECTOR_TABLE).count():,}")

display(
    spark.table(REVIEWS_FOR_VECTOR_TABLE)
    .groupBy("business_name")
    .agg(F.count("*").alias("review_text_count"))
    .orderBy(F.desc("review_text_count"))
)

Created vector-search source table: main.aai510_final_agent.reviews_for_vector_search
Total rows: 1,503


business_name,review_text_count
Parma Cucina Italiana,349
Cesarina,320
Round Table Pizza,100
RoVino The Foodery,50
Ulivo Restaurant,50
Pernicano's Ristorante,50
Villa Capri,50
Mia Trattoria,50
Piacere Mio,50
Il Postino Restaurant,50


### 6.2 Initialize Vector Search Client

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("Vector Search client initialized.")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector Search client initialized.


### 6.3. Configure Vector Search Endpoint and Index

In [0]:
# Vector Search configuration
VECTOR_SEARCH_ENDPOINT_NAME = "restaurant_reviews_vs_endpoint"
VECTOR_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.restaurant_review_text_index"

# Databricks embedding model endpoint
# If this endpoint is not available in your workspace, replace it with the embedding endpoint your course/workspace provides.
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

print("=" * 60)
print("Configuring Vector Search index...")
print("=" * 60)

print(f"Endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
print(f"Index: {VECTOR_SEARCH_INDEX_NAME}")
print(f"Source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Embedding endpoint: {EMBEDDING_MODEL_ENDPOINT_NAME}")

Configuring Vector Search index...
Endpoint: restaurant_reviews_vs_endpoint
Index: main.aai510_final_agent.restaurant_review_text_index
Source table: main.aai510_final_agent.reviews_for_vector_search
Embedding endpoint: databricks-gte-large-en


### 6.4. Create or Connect to Vector Search Endpoint

Create an endpoint/index if needed, otherwise connect to the existing one

In [0]:
# Create Vector Search endpoint if needed

try:
    vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Found existing endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

except Exception:
    print(f"Creating endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    vsc.wait_for_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Created endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

✓ Found existing endpoint: restaurant_reviews_vs_endpoint


### 6.5. Create or Connect to Vector Search Index


In [0]:
# Delete offline / partial Vector Search index

try:
    print(f"Deleting index: {VECTOR_SEARCH_INDEX_NAME}")
    
    vsc.delete_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    print(f"Deleted index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception as e:
    print("Index delete failed or index was already gone.")
    print(e)

Deleting index: main.aai510_final_agent.restaurant_review_text_index
Deleted index: main.aai510_final_agent.restaurant_review_text_index


In [0]:
# Create Delta Sync index if needed

try:
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    print(f"✓ Found existing index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception:
    print(f"Creating index: {VECTOR_SEARCH_INDEX_NAME}")
    
    review_index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME,
        source_table_name=REVIEWS_FOR_VECTOR_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="review_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
        columns_to_sync=[
            "review_id",
            "gmap_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ]
    )
    
    review_index.wait_until_ready()
    print(f"✓ Created index: {VECTOR_SEARCH_INDEX_NAME}")

Creating index: main.aai510_final_agent.restaurant_review_text_index
✓ Created index: main.aai510_final_agent.restaurant_review_text_index


### 6.6. Sync Vector Search Index

In [0]:
# Sync the index after creating or updating the source table

review_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_SEARCH_INDEX_NAME
)

review_index.sync()
review_index.wait_until_ready()

print("✓ Vector Search index is synced and ready.")

✓ Vector Search index is synced and ready.


---
## Pipeline handoff

After this notebook finishes successfully, run `02_agent.ipynb`. That notebook loads the existing tables and vector index, defines the tools, creates the tool-calling agent, and runs tests/evaluation.
